In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import pypsa
import xlsxwriter
import tz_pypsa
import tz_pypsa.wrangle as wrangle
import pandas as pd
# import tz_solve
import plotly.express as px
import plotly.graph_objects as go
from tz_pypsa.model import Model
from tz_pypsa.utils import get_examples
import os
import glob

In [ ]:
n = pypsa.Network()
n.import_from_netcdf("C:/Users/jy/TransitionZero/Google - CFE - Documents/04. Country Specific Vault/Japan/02. Data & Results/Outputs/1_Diagnosis/TP1/Run003/Data/JPN_P2_JPN01/solved_networks/hourly_matching_CFE100_2030.nc")

INFO:pypsa.io:Imported network hourly_matching_CFE100_2030.nc has buses, carriers, generators, links, loads, storage_units


In [7]:
n.storage_units_t.state_of_charge.filter(regex='C&I')

StorageUnit,JPN01 C&I Grid-Batteries
snapshot,
2030-01-01 00:00:00,791.605497
2030-01-01 01:00:00,642.638303
2030-01-01 02:00:00,524.060983
2030-01-01 03:00:00,415.745847
2030-01-01 04:00:00,305.516566
...,...
2030-12-31 19:00:00,663.911000
2030-12-31 20:00:00,751.725177
2030-12-31 21:00:00,842.435804


In [3]:
yearly_df = wrangle.transform_visualiser_yearly_output(n, scenario='CFE')

In [4]:
yearly_df

,Type,Bus,Name,Tech,Vintage,Metric,Value,BusType,Market,Pypsa_Run_Id,Scenario,Year
0,Generator,JPN01,JPN01-Biomass-2023active_exo,Biomass,2023active_exo,Optimal Capacity,419.390000,Brownfield,Unspecified,Unspecified,CFE,2030
1,Generator,JPN01,JPN01-Biomass-2030_endo,Biomass,2030_endo,Optimal Capacity,0.000050,Brownfield,Unspecified,Unspecified,CFE,2030
2,Generator,JPN01,JPN01-Biomass-2030_exo,Biomass,2030_exo,Optimal Capacity,360.000000,Brownfield,Unspecified,Unspecified,CFE,2030
3,Generator,JPN01,JPN01-BlueNH3-2030_endo,Coal & blue NH3: NH3,2030_endo,Optimal Capacity,0.000090,Brownfield,Unspecified,Unspecified,CFE,2030
4,Generator,JPN01,JPN01-BlueNH3-2030_exo,Coal & blue NH3: NH3,2030_exo,Optimal Capacity,140.000000,Brownfield,Unspecified,Unspecified,CFE,2030
...,...,...,...,...,...,...,...,...,...,...,...,...
3726,NaN,JPN01 C&I,NaN,OnshoreWind,NaN,unit_cost_ci_energy,131.205119,Greenfield,Unspecified,Unspecified,CFE,2030
3727,NaN,JPN01 C&I,NaN,Batteries,NaN,unit_cost_ci_energy,0.007554,Greenfield,Unspecified,Unspecified,CFE,2030
3728,NaN,JPN01 C&I,NaN,Grid Imports,NaN,unit_cost_ci_energy,7.174563,Greenfield,Unspecified,Unspecified,CFE,2030
3729,NaN,JPN01 C&I,NaN,Grid Exports,NaN,unit_cost_ci_energy,-7.110963,Greenfield,Unspecified,Unspecified,CFE,2030


In [13]:
def get_ci_parent_emissions(n: pypsa.Network, bus: str) -> float:
    '''Returns hourly emissions in tonnes CO2-eq for the C&I bus
    '''
    ci_parent_generators = n.generators[n.generators.index.str.contains(bus)]
    ci_parent_generators_t = n.generators_t.p[ci_parent_generators.index]
    ci_parent_load = 1/(n.loads_t.p.filter(regex=bus).filter(regex='^(?!.*C&I)'))
    emissions = (
        (
            ci_parent_generators_t
            / ci_parent_generators.efficiency 
            * ci_parent_generators.carrier.map(n.carriers.co2_emissions)
        )
        .sum(axis=1)
    )
    emissions_intensity = emissions * ci_parent_load.squeeze()
    import_flow = n.links_t.p0.filter(regex='C&I').filter(regex='Import').sum(axis=1)
    ci_load = n.loads_t.p.filter(regex='C&I').sum(axis=1)#
    df = pd.DataFrame({
    'snapshot': emissions_intensity.index,
    'emissions_intensity': emissions_intensity.values,
    'import': import_flow.values,
    'ci_load': ci_load.values,
        })
    df['ci_emissions'] = df['emissions_intensity'] * df['import']
    df[['ci_emissions', 'ci_load']].sum

    return df

In [ ]:
get_ci_parent_emissions(n, 'JPN01')

In [15]:
def get_scenario_emission_intensity(n: pypsa.Network, bus: str, units='gCO2/kWh') -> float:
    """
    Calculate the overall emission intensity for a C&I scenario.
    
    Parameters:
    -----------
    n : pypsa.Network
        Solved PyPSA network
    bus : str
        Bus identifier (e.g., 'JPN01')
    units : str
        Units for emission intensity ('gCO2/kWh' or 'tCO2/MWh')
        
    Returns:
    --------
    float
        Emission intensity value
    """
    # Get grid emission intensity for each hour
    ci_parent_generators = n.generators[n.generators.index.str.contains(bus)]
    ci_parent_generators_t = n.generators_t.p[ci_parent_generators.index]
    ci_parent_load = 1/(n.loads_t.p.filter(regex=bus).filter(regex='^(?!.*C&I)'))
    
    # Calculate hourly grid emissions intensity (tCO2/MWh)
    grid_emissions = (
        (
            ci_parent_generators_t
            / ci_parent_generators.efficiency 
            * ci_parent_generators.carrier.map(n.carriers.co2_emissions)
        )
        .sum(axis=1)
    )
    grid_emissions_intensity = grid_emissions * ci_parent_load.squeeze()
    
    # Get C&I imports and calculate total emissions
    ci_imports = n.links_t.p0.filter(regex='C&I').filter(regex='Import').values.flatten()
    total_ci_emissions = (grid_emissions_intensity.values * ci_imports).sum()
    
    # Get total C&I load
    total_ci_load = n.loads_t.p.filter(regex='C&I').sum().sum()
    
    # Calculate emission intensity
    emission_intensity = total_ci_emissions / total_ci_load
    
    # Convert units if needed
    if units == 'gCO2/kWh':
        emission_intensity *= 1000  # tCO2/MWh -> gCO2/kWh
    
    return emission_intensity


In [ ]:
get_scenario_emission_intensity(n, 'JPN01')

In [ ]:
# Example usage: Create a flat dataframe with emission intensity for multiple scenarios
def create_scenario_summary_df(scenarios_dict, bus='JPN01'):
    """
    Create a flat dataframe with emission intensities for multiple scenarios.
    
    Parameters:
    -----------
    scenarios_dict : dict
        Dictionary with scenario names as keys and pypsa.Network objects as values
    bus : str
        Bus identifier
        
    Returns:
    --------
    pd.DataFrame
        Flat dataframe with scenario data
    """
    results = []
    
    for scenario_name, network in scenarios_dict.items():
        emission_intensity = get_scenario_emission_intensity(network, bus, units='gCO2/kWh')
        
        # You can add other metrics here too
        total_ci_load = network.loads_t.p.filter(regex='C&I').sum().sum()
        total_ci_imports = network.links_t.p0.filter(regex='C&I').filter(regex='Import').sum().sum()
        
        results.append({
            'scenario': scenario_name,
            'bus': bus,
            'emission_intensity_gCO2_per_kWh': emission_intensity,
            'total_ci_load_MWh': total_ci_load,
            'total_ci_imports_MWh': total_ci_imports,
            'self_sufficiency_ratio': 1 - (total_ci_imports / total_ci_load) if total_ci_load > 0 else 0
        })
    
    return pd.DataFrame(results)

# Example with single scenario
emission_intensity = get_scenario_emission_intensity(n, 'JPN01')
print(f"Emission intensity: {emission_intensity:.2f} gCO2/kWh")

# Example dataframe for single scenario
single_scenario_df = pd.DataFrame({
    'scenario': ['annual_matching_RES100_2030'],
    'bus': ['JPN01'],
    'emission_intensity_gCO2_per_kWh': [emission_intensity],
    'total_ci_load_MWh': [n.loads_t.p.filter(regex='C&I').sum().sum()],
    'year': [2030]
})

print("\nSingle scenario dataframe:")
print(single_scenario_df)


In [4]:
def get_ci_parent_emissions(n: pypsa.Network, nodes_with_ci_loads) -> float:
    '''Returns hourly emissions in tonnes CO2-eq for the C&I bus
    '''
    ci_parent_generators = n.generators[n.generators.index.str.contains(nodes_with_ci_loads)]
    ci_parent_generators_t = n.generators_t.p[ci_parent_generators.index]
    ci_parent_load = 1/(n.loads_t.p.filter(regex=nodes_with_ci_loads).filter(regex='^(?!.*C&I)'))
    emissions = (
        (
            ci_parent_generators_t
            / ci_parent_generators.efficiency 
            * ci_parent_generators.carrier.map(n.carriers.co2_emissions)
        )
        .sum(axis=1)
    )
    emissions_intensity = emissions * ci_parent_load.squeeze()
    return emissions_intensity

In [ ]:
get_ci_parent_emissions(n, 'JPN01')

In [38]:
ci_emissions =(
        np.sum(
            n.links_t.p0.filter(regex='C&I').filter(regex='Import').values.flatten() @ np.array(get_ci_parent_emissions(n, 'JPN01'))
        ) 
)

ci_load = n.loads_t.p.filter(regex='C&I').sum(axis=1).sum()

ci_emission_rate = ci_emissions/ci_load

In [ ]:
n.links_t.p0.filter(regex='C&I').filter(regex='Import').v

In [ ]:
ci_emission_rate

In [ ]:
df = pd.DataFrame({'Value': ci_emissions})

In [43]:
ci_emissions = (np.array(get_ci_parent_emissions(n, 'JPN01')) * n.links_t.p0.filter(regex='C&I').filter(regex='Import').values.flatten()).sum()

In [ ]:
ci_emissions

In [18]:
ci_load = n.loads_t.p.filter(regex='C&I').sum()

In [ ]:
ci_emissions/ci_load

In [ ]:
n.links_t.p0.filter(regex='C&I').filter(regex='Import').values.flatten()

In [ ]:
get_ci_parent_emissions(n, 'JPN01')

In [ ]:
ci_emissions/ci_load

In [ ]:
n.links_t.p0.filter(regex='C&I').filter(regex='Import').sum(axis=0)

In [ ]:
n.links_t.p0.filter(regex='C&I').filter(regex='Export').sum(axis=0)

In [ ]:
diff = 530586-554.278062
diff

In [ ]:
n.links

In [4]:
def plot_ci_emission_rate_by_scenario(solved_networks, run):
    """
    Plot C&I emission rate [gCO2/kWh] by scenario.
    """
    # ------------------------------------------------------------------
    # C&I EMISSION RATE

    print('Creating C&I emission rate by scenario plot')

    ci_emissions = (
        pd
        .DataFrame({
            'name' : [k for k in solved_networks.keys()],
            'load' : [solved_networks[k].loads_t.p_set.filter(regex='C&I').sum().sum() for k in solved_networks.keys()],
            'emissions' : [
                np.sum(
                    solved_networks[k].links_t.p0.filter(regex='C&I').filter(regex='Import').values.flatten() @ np.array(GetGridCFE(solved_networks[k], ci_identifier='C&I'))
                ) 
                for k in solved_networks.keys()
            ],
        })
    )

    ci_emissions['emission_rate'] = (ci_emissions['emissions'] / ci_emissions['load']) * 1000 # tCO2 / MWh -> gCO2 / kWh

    res = (
        ci_emissions
        .loc[ci_emissions['Scenario'] == '100% RES']
        .pivot_table(index='Scenario', values='emission_rate')
        .reset_index()
    )

    cfe = (
        ci_emissions
        .loc[ci_emissions['Scenario'].str.contains('CFE')]
        .pivot_table(index='CFE Score', values='emission_rate')
        .reset_index()
    )

    return res, cfe

In [ ]:
plot_ci_emission_rate_by_scenario(n, run)